# Florence-2 Fine-tuning on DocVQA

Notebook for Google Colab A100. Data is pulled from Hugging Face Hub via `src/colab_setup.py`, and checkpoints are uploaded back to Hugging Face Hub while metrics and tables are logged to Comet.

Prompt format follows the official Florence-2 model card pattern: `prompt = task_prompt + text_input`, then `processor(text=prompt, images=image, ...)`. For VQA, this notebook uses `task_prompt = "<VQA>"` and appends the question after it.

Reference: https://huggingface.co/docs/transformers/model_doc/florence2

## 0. Setup

In [3]:
%pip install -q huggingface_hub comet_ml python-dotenv timm einops

In [4]:
!cd /content/course_work2026 && git fetch origin && git reset --hard origin/main


HEAD is now at 22f7d9c Fix Colab bootstrap and update HF/Comet notebook setup


In [ ]:
import os



os.environ["HF_DATASET_REPO"] = "sk3feel/docvqa-privacy-data"
os.environ["HF_MODEL_REPO"] = "sk3feel/docvqa-privacy-checkpoints"
os.environ["COMET_WORKSPACE"] = "scfeel"
os.environ["COMET_PROJECT_NAME"] = "qwen3-1"
os.environ["COURSE_WORK2026_REPO_URL"] = "https://github.com/sk3feel/hidden-data-reproduction-multimodal.git"


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists() and (candidate / 'requirements.txt').exists():
            return candidate
    return None


def get_colab_secret(name: str) -> str | None:
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.getenv(name)


project_root = find_project_root(Path.cwd())
repo_url = get_colab_secret('COURSE_WORK2026_REPO_URL') or 'https://github.com/sk3feel/hidden-data-reproduction-multimodal.git'

if project_root is None:
    clone_dir = Path('/content/course_work2026')
    if not (clone_dir / '.git').exists():
        if not repo_url:
            raise RuntimeError('Could not detect the repository locally and COURSE_WORK2026_REPO_URL is not set in Colab secrets.')
        subprocess.run(['git', 'clone', repo_url, str(clone_dir)], check=True)
    else:
        subprocess.run(['git', '-C', str(clone_dir), 'pull', '--ff-only'], check=True)
    project_root = clone_dir

sys.path.insert(0, str(project_root / 'src'))

from colab_setup import setup_colab

setup_summary, experiment = setup_colab(repo_url=repo_url)
experiment.set_name('florence2-finetune')

PROJECT_ROOT = Path(setup_summary['project_root'])
ARTIFACTS_ROOT = PROJECT_ROOT / 'artifacts'
TRAIN_JSONL = ARTIFACTS_ROOT / 'finetuning_generative' / 'train_florence2.jsonl'
VALIDATION_JSONL = ARTIFACTS_ROOT / 'finetuning_generative' / 'validation_florence2.jsonl'
OUTPUT_ROOT = ARTIFACTS_ROOT / 'finetuning_generative' / 'florence2_base_colab'
CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
TRAIN_LOSS_CSV = OUTPUT_ROOT / 'train_loss.csv'
SANITY_RESULTS_CSV = OUTPUT_ROOT / 'sanity_check_results.csv'

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(json.dumps(setup_summary, ensure_ascii=False, indent=2))
print({'project_root': str(PROJECT_ROOT), 'train_jsonl_exists': TRAIN_JSONL.exists(), 'validation_jsonl_exists': VALIDATION_JSONL.exists()})

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


artifacts_bundle.tar.gz:   0%|          | 0.00/3.04G [00:00<?, ?B/s]

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/scfeel/qwen3-1/f08645d5cfc9402d9ccfbd4c1d88e67e



{
  "project_root": "/content/course_work2026",
  "dataset_repo_id": "sk3feel/docvqa-privacy-data",
  "dataset_transport": "hf_hub_download + tar.gz bundle",
  "download_summary": {
    "repo_id": "sk3feel/docvqa-privacy-data",
    "local_dir": "/content/course_work2026/artifacts",
    "archive_filename": "artifacts_bundle.tar.gz",
    "extracted_paths": {
      "benchmark_train": "/content/course_work2026/artifacts/docqa_recovery/benchmark_train",
      "benchmark": "/content/course_work2026/artifacts/docqa_recovery/benchmark",
      "finetuning_generative": "/content/course_work2026/artifacts/finetuning_generative"
    }
  },
  "comet_workspace": "scfeel",
  "comet_project_name": "qwen3-1",
  "comet_experiment_key": "f08645d5cfc9402d9ccfbd4c1d88e67e",
  "comet_experiment_url": null
}
{'project_root': '/content/course_work2026', 'train_jsonl_exists': True, 'validation_jsonl_exists': True}


COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.


## 1. Load Data

In [7]:
import re
import time

import comet_ml
import pandas as pd
import torch
from IPython.display import display
from PIL import Image, ImageFile
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoProcessor, Florence2ForConditionalGeneration
from huggingface_hub import HfApi

MODEL_NAME = 'florence-community/Florence-2-base'
ImageFile.LOAD_TRUNCATED_IMAGES = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_MIXED_PRECISION = False
amp_dtype = None
use_grad_scaler = False
HF_CHECKPOINT_REPO = os.getenv('HF_MODEL_REPO') or 'sk3feel/docvqa-privacy-checkpoints'
api = HfApi()
api.create_repo(repo_id=HF_CHECKPOINT_REPO, repo_type='model', private=True, exist_ok=True)

QUESTION_RE = re.compile(r'<Question>(.*?)</Question>', flags=re.IGNORECASE | re.DOTALL)


def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def extract_question(task_prompt: str) -> str:
    match = QUESTION_RE.search(task_prompt)
    if match:
        return match.group(1).strip()
    return task_prompt.replace('<VQA>', '').strip()


def resolve_image_path(raw_path: str, split: str) -> Path:
    original = Path(raw_path)
    if original.exists():
        return original
    image_name = raw_path.replace('\\', '/').split('/')[-1]
    benchmark_dir = 'benchmark_train' if split == 'train' else 'benchmark'
    candidate = ARTIFACTS_ROOT / 'docqa_recovery' / benchmark_dir / 'images' / 'original' / image_name
    if candidate.exists():
        return candidate
    fallback = next((ARTIFACTS_ROOT / 'docqa_recovery').rglob(image_name), None)
    if fallback is not None:
        return fallback
    raise FileNotFoundError(f'Cannot resolve image path for {raw_path}')


def load_rgb_image(image_path: str | Path) -> Image.Image:
    with Image.open(image_path) as image:
        return image.convert('RGB').copy()


def prepare_florence_rows(rows: list[dict]) -> list[dict]:
    prepared = []
    for row in rows:
        question = extract_question(str(row['task_prompt']))
        image_path = resolve_image_path(str(row['image_path']), str(row['split']))
        prepared.append({
            'example_id': row['example_id'],
            'split': row['split'],
            'question': question,
            'task_prompt': '<VQA>',
            'task_prompt_with_question': f'<VQA>{question}',
            'answer': str(row['answer']).strip(),
            'image_path': str(image_path),
        })
    return prepared


train_rows = prepare_florence_rows(load_jsonl(TRAIN_JSONL))
validation_rows = prepare_florence_rows(load_jsonl(VALIDATION_JSONL))

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = Florence2ForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
model.config.use_cache = False

display(pd.DataFrame(train_rows[:3])[['example_id', 'question', 'task_prompt_with_question', 'answer', 'image_path']])

experiment.log_parameter('train_size', len(train_rows))
experiment.log_parameters({
    'validation_size': len(validation_rows),
    'model_name': MODEL_NAME,
    'task_prompt': '<VQA>',
    'checkpoint_repo': HF_CHECKPOINT_REPO,
})

print({'device': str(device), 'use_mixed_precision': USE_MIXED_PRECISION, 'amp_dtype': str(amp_dtype), 'use_grad_scaler': use_grad_scaler, 'train_examples': len(train_rows), 'validation_examples': len(validation_rows)})

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/665 [00:00<?, ?it/s]

Florence2ForConditionalGeneration LOAD REPORT from: florence-community/Florence-2-base
Key            | Status  | 
---------------+---------+-
lm_head.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,example_id,question,task_prompt_with_question,answer,image_path
0,57cadc4b97472b724c6a63309529ca3a,What is the type of organization?,<VQA>What is the type of organization?,Corporation,/content/course_work2026/artifacts/docqa_recov...
1,54053bc0e55fb6b4b684bbbc290dcf40,what is the price at bottom of the page ?,<VQA>what is the price at bottom of the page ?,$1.90,/content/course_work2026/artifacts/docqa_recov...
2,27a10a952656f5096c780135ad71fa92,Who has MCH as area of special emphasis and fr...,<VQA>Who has MCH as area of special emphasis a...,"BANTA, JAMES E.",/content/course_work2026/artifacts/docqa_recov...


{'device': 'cuda', 'amp_dtype': 'torch.bfloat16', 'use_grad_scaler': False, 'train_examples': 800, 'validation_examples': 1612}


## 2. Dataset and DataLoader

In [8]:
class FlorenceVQADataset(Dataset):
    def __init__(self, rows: list[dict], processor: AutoProcessor):
        self.rows = rows
        self.processor = processor
        self.tokenizer = processor.tokenizer

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int) -> dict:
        row = self.rows[idx]
        image = load_rgb_image(row['image_path'])
        inputs = self.processor(
            images=image,
            text=row['task_prompt_with_question'],
            return_tensors='pt',
        )
        labels = self.tokenizer(
            row['answer'],
            return_tensors='pt',
            add_special_tokens=True,
        ).input_ids.squeeze(0)
        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'labels': labels,
            'question': row['question'],
            'answer': row['answer'],
            'image_path': row['image_path'],
            'example_id': row['example_id'],
        }


def make_collate_fn(processor: AutoProcessor):
    tokenizer = processor.tokenizer
    pad_token_id = tokenizer.pad_token_id
    if pad_token_id is None:
        pad_token_id = tokenizer.eos_token_id

    def collate_fn(batch: list[dict]) -> dict:
        input_ids = torch.nn.utils.rnn.pad_sequence(
            [item['input_ids'] for item in batch],
            batch_first=True,
            padding_value=pad_token_id,
        )
        attention_mask = torch.nn.utils.rnn.pad_sequence(
            [item['attention_mask'] for item in batch],
            batch_first=True,
            padding_value=0,
        )
        labels = torch.nn.utils.rnn.pad_sequence(
            [item['labels'] for item in batch],
            batch_first=True,
            padding_value=-100,
        )
        pixel_values = torch.stack([item['pixel_values'] for item in batch])
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'pixel_values': pixel_values,
            'labels': labels,
            'question': [item['question'] for item in batch],
            'answer': [item['answer'] for item in batch],
            'image_path': [item['image_path'] for item in batch],
            'example_id': [item['example_id'] for item in batch],
        }

    return collate_fn


train_dataset = FlorenceVQADataset(train_rows, processor)
validation_dataset = FlorenceVQADataset(validation_rows, processor)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    collate_fn=make_collate_fn(processor),
)

sample_batch = next(iter(train_loader))
print({k: tuple(v.shape) for k, v in sample_batch.items() if hasattr(v, 'shape')})

{'input_ids': (4, 598), 'attention_mask': (4, 598), 'pixel_values': (4, 3, 768, 768), 'labels': (4, 10)}


## 3. Fine-tuning

In [9]:
NUM_EPOCHS = 50
LEARNING_RATE = 1e-5
MAX_GRAD_NORM = 1.0
SAVE_EPOCHS = {1, 3, 10, 30, 50}

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
scaler = torch.amp.GradScaler('cuda', enabled=use_grad_scaler) if torch.cuda.is_available() else None
train_history = []


def save_checkpoint(model, processor, optimizer, epoch: int, mean_loss: float) -> Path:
    checkpoint_path = CHECKPOINT_DIR / f'epoch_{epoch}'
    checkpoint_path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(checkpoint_path)
    processor.save_pretrained(checkpoint_path)
    torch.save(
        {
            'epoch': epoch,
            'optimizer_state_dict': optimizer.state_dict(),
            'mean_loss': mean_loss,
        },
        checkpoint_path / 'trainer_state.pt',
    )
    return checkpoint_path


for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_losses = []
    skipped_batches = 0
    epoch_start = time.time()

    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS}', leave=False)
    for batch in progress:
        optimizer.zero_grad(set_to_none=True)

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        pixel_values = batch['pixel_values'].to(device=device)
        labels = batch['labels'].to(device)

        with torch.amp.autocast(device_type=device.type, enabled=USE_MIXED_PRECISION and torch.cuda.is_available(), dtype=amp_dtype):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                labels=labels,
            )
            loss = outputs.loss

        if not torch.isfinite(loss):
            skipped_batches += 1
            progress.write(f"Skipping non-finite loss at epoch {epoch}, examples={batch['example_id']}")
            optimizer.zero_grad(set_to_none=True)
            continue

        if scaler is not None and scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()

        loss_value = float(loss.detach().item())
        epoch_losses.append(loss_value)
        progress.set_postfix({'loss': f'{loss_value:.4f}'})

    epoch_seconds = time.time() - epoch_start
    mean_loss = float('nan') if not epoch_losses else (sum(epoch_losses) / len(epoch_losses))
    train_history.append({
        'epoch': epoch,
        'train_loss': mean_loss,
        'epoch_seconds': epoch_seconds,
        'skipped_batches': skipped_batches,
    })
    pd.DataFrame(train_history).to_csv(TRAIN_LOSS_CSV, index=False)

    experiment.log_metric('train_loss', mean_loss, epoch=epoch)
    experiment.log_metric('skipped_batches', skipped_batches, epoch=epoch)
    print(f'Epoch {epoch:02d} | loss={mean_loss:.4f} | skipped={skipped_batches} | time={epoch_seconds:.1f}s')

    if epoch in SAVE_EPOCHS and epoch_losses and mean_loss == mean_loss and mean_loss != float('inf'):
        checkpoint_path = save_checkpoint(model, processor, optimizer, epoch, mean_loss)
        api.upload_folder(
            folder_path=str(checkpoint_path),
            repo_id=HF_CHECKPOINT_REPO,
            repo_type='model',
            path_in_repo=f'florence2/epoch_{epoch}',
        )
        experiment.log_metric('checkpoint_saved', epoch, epoch=epoch)
        print(f'  saved and uploaded checkpoint: {checkpoint_path}')
    elif epoch in SAVE_EPOCHS:
        print(f'  skipped checkpoint save for epoch {epoch} because mean loss is non-finite')

print(f'Train log saved to: {TRAIN_LOSS_CSV}')

Epoch 1/50:   0%|          | 0/200 [00:00<?, ?it/s]

Skipping non-finite loss at epoch 1, examples=['a9f6893e24f83e90c789401f0d0bf2f0', '803bcf051cbe1b8ddefdc8147c37860e', '87e4a2201408645c02a5a6e4cf158cdd', '0b51cfe60f766836dabf83ee367920ba']
Skipping non-finite loss at epoch 1, examples=['d1ad053d7bc6b244b9593bb5fa0b5c80', '9d59a9238abcd90da0cc57e25f262197', '7c7da3c3b5bdcef593d4c6c52973e295', 'fc7ace70eee057988781872866b6aa06']
Skipping non-finite loss at epoch 1, examples=['37e540a521d386369c234947c8f59ff4', '90c77cc742e5e6bd09fb73ea933d4bea', '52c8daab602fe06c422aea94aa56b0ab', '63412748798b3a85cc2558039e0e6384']
Skipping non-finite loss at epoch 1, examples=['86f8c6bdf812361bf9e4650b9dd10ca8', 'ca4947a5e36148be8ae2e696583d32c6', '4719c41b9f435fc9ecc0afb520378f3c', 'c8586d347d30affa9392177bddc88341']
Skipping non-finite loss at epoch 1, examples=['780edb7cb5737e27cd7d95386bb06184', 'bced93a179e869027b7e87996b16cab0', '1810cb6007fa181ad3972f9ecc21af16', '1711bd04b00cdf430bc88659f623b13c']
Skipping non-finite loss at epoch 1, examples

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Epoch 2/50:   0%|          | 0/200 [00:00<?, ?it/s]

Skipping non-finite loss at epoch 2, examples=['cdc4331e4ffb8970d73ed74675b180f4', '0af139e88d5ac24e20e4946aab4fe133', '9bf80612a8b76c5e9fd69d1b96e3fe05', '281e2bd7d3ccb0cd79db8abfbeca5b0d']
Skipping non-finite loss at epoch 2, examples=['73ee3696b928fe1db9ae142cf879bc61', 'bb55d31ecc50c32ebcec3d37bf009c6b', 'd3094559be62ff1b2038799ee1135b72', 'c218ca6a2b65a63f55edbadecfe98dbb']
Skipping non-finite loss at epoch 2, examples=['29a58b48a9d40c36d5ca95d23a20601b', '42ab673def0b38940e0461dda88870d3', '60c48e90c7e3e66cfab6930b19d094dc', '349cd1a8c7f38aaef160667505bc3b23']
Skipping non-finite loss at epoch 2, examples=['63d49d6a8a11e0feb9486b37f53074ba', '2419d6b519668f0463789fff5d4f4a75', '2b892afb6e523e49484b77edd10f2db5', '23252c59768d9bccf038340889f7a8cd']
Skipping non-finite loss at epoch 2, examples=['013626f16dc4cb2ecb15c325a9dc6a6c', '2b483403904529e85b3fb219d1c1cde3', '3c831ed40e70ddea61c93cd0caff3354', '94bd18b2b8193b968ad0f0b08a6978cb']
Skipping non-finite loss at epoch 2, examples

Epoch 3/50:   0%|          | 0/200 [00:00<?, ?it/s]

Skipping non-finite loss at epoch 3, examples=['cd6ec7abde644587b986596c9949d381', '1e9088b88903fe092a1685df742cb220', '53f4a4c016bd5bd6c4721c7fc13d85a3', '00f603f4ecd53e849086aebeff684122']
Skipping non-finite loss at epoch 3, examples=['709eaaf0bb5449269c6b7f5c843aa6d5', 'fbbbe2c061a958396cb6b6af048ffb3d', 'd6ad09ebe6250a80a8e07285eb2c3144', '42b005ad63480d2a00e63cfca5458e67']
Skipping non-finite loss at epoch 3, examples=['a1ba09fc72b650e9040db9d8d1075f33', '41aaaad95f140bf71120e8db65d2a5ab', '61ead4c681c1f4d4aca7bb6b975d9a6b', '7338f228f353a189efb6b0714f472884']
Skipping non-finite loss at epoch 3, examples=['ca00e6bcb6b94dd5052fbecbd1524bc3', 'f2856cb337dc31265a425771949fa82b', '9dd2d5fdc74ddf11b45eea8c083177a1', 'b6d45cac7c4cf4e6e823718f73285ccb']
Skipping non-finite loss at epoch 3, examples=['e27fb575db929ed16b3428242cc545c8', 'ba6d98318c504587b1a81cfc1dd15b4d', 'cce3a28a66439a85d4a46f663e222fc8', '6a73db68fdcf01b3281f31d015b6f35d']
Skipping non-finite loss at epoch 3, examples

KeyboardInterrupt: 

## 4. Sanity Check

In [ ]:
def normalize_text(text: str) -> str:
    return ' '.join(str(text).strip().lower().split())


def sanitize_prediction_text(text: str) -> str:
    text = str(text or '')
    text = re.sub(r'<[^>]+>', ' ', text)
    text = ' '.join(text.replace('\n', ' ').split()).strip()
    return text


def exact_match(gold: str, pred: str) -> float:
    return float(normalize_text(gold) == normalize_text(pred))


def decode_prediction(processor, generated_ids, image: Image.Image, task_prompt: str) -> str:
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    if hasattr(processor, 'post_process_generation'):
        try:
            parsed = processor.post_process_generation(
                generated_text,
                task=task_prompt,
                image_size=(image.width, image.height),
            )
            if isinstance(parsed, dict):
                value = parsed.get(task_prompt)
                if isinstance(value, str):
                    return sanitize_prediction_text(value)
            elif isinstance(parsed, str):
                return sanitize_prediction_text(parsed)
        except Exception:
            pass
    decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return sanitize_prediction_text(decoded)


def run_generation(model, processor, rows: list[dict], split_name: str, limit: int = 100) -> tuple[pd.DataFrame, float]:
    model.eval()
    results = []
    sample_rows = rows[:limit]

    for row in tqdm(sample_rows, desc=f'Sanity {split_name}'):
        image = load_rgb_image(row['image_path'])
        inputs = processor(
            images=image,
            text=row['task_prompt_with_question'],
            return_tensors='pt',
        )
        input_ids = inputs['input_ids'].to(device)
        attention_mask = inputs['attention_mask'].to(device)
        pixel_values = inputs['pixel_values'].to(device=device)

        with torch.no_grad():
            generated = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                max_new_tokens=50,
                num_beams=3,
                do_sample=False,
            )

        prediction = decode_prediction(processor, generated, image=image, task_prompt=row['task_prompt'])
        results.append({
            'split': split_name,
            'example_id': row['example_id'],
            'question': row['question'],
            'gold': row['answer'],
            'prediction': prediction,
            'exact_match': exact_match(row['answer'], prediction),
        })

    result_df = pd.DataFrame(results)
    em = float(result_df['exact_match'].mean()) if not result_df.empty else 0.0
    return result_df, em


epoch_50_dir = CHECKPOINT_DIR / 'epoch_50'
if not epoch_50_dir.exists():
    raise FileNotFoundError(f'Checkpoint not found: {epoch_50_dir}')

sanity_processor = AutoProcessor.from_pretrained(epoch_50_dir)
sanity_model = Florence2ForConditionalGeneration.from_pretrained(epoch_50_dir).to(device)
sanity_model.eval()

seen_results, em_seen = run_generation(sanity_model, sanity_processor, train_rows, 'seen_train', limit=100)
unseen_results, em_unseen = run_generation(sanity_model, sanity_processor, validation_rows, 'unseen_validation', limit=100)

all_results = pd.concat([seen_results, unseen_results], ignore_index=True)
all_results.to_csv(SANITY_RESULTS_CSV, index=False)

experiment.log_metric('em_seen', em_seen)
experiment.log_metric('em_unseen', em_unseen)
experiment.log_table('sanity_check.csv', all_results)

print({'em_seen': em_seen, 'em_unseen': em_unseen, 'sanity_results_csv': str(SANITY_RESULTS_CSV)})
display(all_results[['split', 'question', 'gold', 'prediction', 'exact_match']].head(10))

## 5. Finish

In [ ]:
experiment.end()
print('Comet experiment ended.')